In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as _sum, desc, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("UserAffinityAggregation") \
    .getOrCreate()

In [2]:
df = spark.read.csv("/home/jovyan/work/ecommerce_logs.csv", header=True, inferSchema=True)

print(df.count())
df.show(5)

10831817
+-------------------+---------------+---------+----------+----------+------+-------------+--------------------+--------------------+
|          timestamp|     session_id|  user_id|event_type|product_id| price|     referrer|       user_metadata|    product_metadata|
+-------------------+---------------+---------+----------+----------+------+-------------+--------------------+--------------------+
|2026-02-21 08:33:36|SESS_4007a38849|User_2686|      view| ITEM_3306| 754.5|instagram.com|"{""device"": ""M...|  ""tier"": ""Gold""|
|2026-02-21 08:34:02|SESS_4007a38849|User_2686|      cart| ITEM_4021|654.44|instagram.com|"{""device"": ""M...|  ""tier"": ""Gold""|
|2026-02-21 08:35:42|SESS_4007a38849|User_2686|      view| ITEM_1241|608.71|instagram.com|"{""device"": ""M...|  ""tier"": ""Gold""|
|2026-02-21 08:41:33|SESS_4007a38849|User_2686|      view| ITEM_4064|396.02|instagram.com|                NULL|                NULL|
|2026-02-21 08:43:56|SESS_4007a38849|User_2686|      view| I

In [5]:
from pyspark.sql.functions import regexp_extract, col

df = df.withColumn(
    "category",
    regexp_extract("product_metadata", '""category"":\\s*""([^""]+)""', 1)
)

In [6]:
df.select("product_metadata", "category").show(20, False)

+----------------------------+--------+
|product_metadata            |category|
+----------------------------+--------+
| ""tier"": ""Gold""         |        |
| ""tier"": ""Gold""         |        |
| ""tier"": ""Gold""         |        |
|NULL                        |NULL    |
|"{""category"": ""Clothing""|Clothing|
| ""tier"": ""Gold""         |        |
| ""tier"": ""Gold""         |        |
| ""tier"": ""Gold""         |        |
| ""tier"": ""Gold""         |        |
| ""tier"": ""Platinum""     |        |
| ""tier"": ""Platinum""     |        |
| ""tier"": ""Platinum""     |        |
|"{""category"": ""Toys""    |Toys    |
| ""tier"": ""Silver""       |        |
| ""tier"": ""Silver""       |        |
| ""tier"": ""Silver""       |        |
|"{""category"": ""Books""   |Books   |
| ""tier"": ""Silver""       |        |
| ""tier"": ""Silver""       |        |
|"{""category"": ""Home""    |Home    |
+----------------------------+--------+
only showing top 20 rows



In [12]:
df_valid = df.filter(
    (col("category").isNotNull()) & (col("category") != "")
)

In [13]:
df_weighted = df_valid.withColumn(
    "weight",
    when(col("event_type") == "view", 1)
    .when(col("event_type") == "cart", 3)
    .when(col("event_type") == "purchase", 5)
    .otherwise(0)
)

In [14]:
df_agg = df_weighted.groupBy("user_id", "category") \
    .agg(_sum("weight").alias("score"))

In [15]:
window = Window.partitionBy("user_id").orderBy(desc("score"))

df_ranked = df_agg.withColumn(
    "rank",
    row_number().over(window)
)

In [16]:
df_ranked.orderBy("user_id", "rank").show(50, False)

+----------+-----------+-----+----+
|user_id   |category   |score|rank|
+----------+-----------+-----+----+
|User_0    |Clothing   |17   |1   |
|User_0    |Electronics|13   |2   |
|User_0    |Toys       |10   |3   |
|User_0    |Home       |9    |4   |
|User_0    |Books      |6    |5   |
|User_1    |Electronics|18   |1   |
|User_1    |Home       |17   |2   |
|User_1    |Clothing   |11   |3   |
|User_1    |Books      |11   |4   |
|User_1    |Toys       |9    |5   |
|User_10   |Books      |16   |1   |
|User_10   |Toys       |15   |2   |
|User_10   |Home       |12   |3   |
|User_10   |Electronics|7    |4   |
|User_10   |Clothing   |3    |5   |
|User_100  |Electronics|13   |1   |
|User_100  |Home       |13   |2   |
|User_100  |Books      |13   |3   |
|User_100  |Clothing   |5    |4   |
|User_100  |Toys       |3    |5   |
|User_1000 |Books      |17   |1   |
|User_1000 |Clothing   |16   |2   |
|User_1000 |Toys       |14   |3   |
|User_1000 |Home       |10   |4   |
|User_1000 |Electronics|9   

In [17]:
df_top1 = df_ranked.filter(col("rank") == 1)

df_top1.show(20, False)

+----------+-----------+-----+----+
|user_id   |category   |score|rank|
+----------+-----------+-----+----+
|User_1    |Electronics|18   |1   |
|User_10   |Books      |16   |1   |
|User_100  |Electronics|13   |1   |
|User_1000 |Books      |17   |1   |
|User_10000|Electronics|22   |1   |
|User_10001|Clothing   |16   |1   |
|User_10002|Books      |17   |1   |
|User_1001 |Home       |25   |1   |
|User_10011|Toys       |17   |1   |
|User_10012|Books      |15   |1   |
|User_10017|Books      |24   |1   |
|User_10019|Clothing   |15   |1   |
|User_10023|Books      |14   |1   |
|User_10025|Books      |35   |1   |
|User_10027|Home       |13   |1   |
|User_10029|Home       |25   |1   |
|User_10030|Electronics|18   |1   |
|User_10031|Books      |17   |1   |
|User_10032|Home       |18   |1   |
|User_10037|Electronics|15   |1   |
+----------+-----------+-----+----+
only showing top 20 rows



In [21]:
top_k = df_ranked.filter(col("rank") <= 3)

In [22]:
top_k.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/user_affinity")